# Previsao Climatica - pipeline WORCAP/INPE

Roda o pipeline completo (build_dataset -> train -> predict) usando a GPU e os dados da competicao ja montados pelo Kaggle em `/kaggle/input/`.

**Antes de rodar:**
1. Settings (painel direito) -> Accelerator -> **GPU T4 x2** (ou P100).
2. Settings -> Internet -> **On** (precisa pra clonar o repo).
3. Add Data -> anexa o dataset da competicao (se ainda nao estiver anexado).
4. Se o repo do GitHub for **privado**, cria um Personal Access Token (read-only, repo scope) e cola em `GITHUB_TOKEN` na celula abaixo, ou (mais simples) sobe o repo como um Kaggle Dataset em vez de git clone.

In [ ]:
# confirma o nome exato da pasta do dataset da competicao
!ls /kaggle/input/

In [ ]:
GITHUB_REPO = "https://github.com/ThiagoLange/Previsao_Climatica.git"
GITHUB_TOKEN = ""  # preenche so se o repo for privado: "https://<token>@github.com/..."

import os
clone_url = GITHUB_REPO.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else GITHUB_REPO

!rm -rf Previsao_Climatica
!git clone --depth 1 {clone_url}
%cd Previsao_Climatica

In [ ]:
# a imagem do Kaggle ja vem com xgboost/pandas/xarray/torch com CUDA -- so garante versoes/pacotes que podem faltar
!pip install -q pyarrow h5netcdf netcdf4 xgboost --upgrade

In [ ]:
# ajusta o nome da pasta se for diferente do que apareceu no `ls /kaggle/input/` acima
COMPETITION_INPUT = "/kaggle/input/previsao-climatica-de-precipitacao-sobre-a-america-do-sul"

!ln -sfn {COMPETITION_INPUT} data
!ls -la data/

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Baseline climatologia (referencia a bater)

In [ ]:
!python -m src.baseline_climatology --split holdout

## 2. Build dataset (holdout) -- Kaggle tem bem mais RAM, pode tentar `--spatial-stride 1` (resolucao cheia) primeiro; se estourar, cai pra 2

In [ ]:
!python -m src.build_dataset --split holdout --spatial-stride 1

## 3. Treina no holdout (GPU) e ve o RMSE vs climatologia

In [ ]:
!python -m src.train --split holdout --device cuda

## 4. Build dataset full + treino full + predicao
So roda depois de confirmar que o holdout acima ficou bom (RMSE menor que a climatologia).

In [ ]:
!python -m src.build_dataset --split full --spatial-stride 1

In [ ]:
!python -m src.train --split full --device cuda --n-estimators 2000

In [ ]:
!mkdir -p /kaggle/working/submissions
!python -m src.predict --split full --output /kaggle/working/submission.csv

## 5. Submeter
`/kaggle/working/submission.csv` fica disponivel na aba **Output** do notebook -- da pra clicar em **Submit to Competition** direto dali, sem precisar baixar/subir manualmente.